# Watching SymPy integrate, step by step

`sympy.integrals.manualintegrate.integral_steps` explains how SymPy would
integrate something the way a person would: by parts, by substitution, term
by term.  What it returns is a **tree of rules**, not a derivation you can
read - and the history viewer of `sympy-editor` wants a **list of
expressions**, each with a word about what produced it.

This notebook flattens the one into the other.  It is an example and nothing
more: `sympy_editor` knows nothing about `manualintegrate`, and this code
lives here, in the notebook, not in the library.  Anything that can produce
a list of expressions can be shown the same way.

In [1]:
from sympy import Integral, Symbol, atan, cos, exp, log, sin, sqrt
from sympy.integrals.manualintegrate import DontKnowRule, Rule, URule, integral_steps

from sympy_editor import History, display_history

x = Symbol("x")
integral_steps(x * sin(x), x)

PartsRule(integrand=x*sin(x), variable=x, u=x, dv=sin(x), v_step=SinRule(integrand=sin(x), variable=x), second_step=ConstantTimesRule(integrand=-cos(x), variable=x, constant=-1, other=cos(x), substep=CosRule(integrand=cos(x), variable=x)))

## One rule at a time

A rule knows how to integrate one thing, and holds the sub-rules for whatever
it leaves behind: `PartsRule` above has a `v_step` and a `second_step`.
Calling `.eval()` on it does the whole job at once - which is exactly what we
do *not* want to see.

So: **evaluate a rule with its sub-rules replaced by `DontKnowRule`**, which
evaluates to an unevaluated `Integral`.  That gives what this rule alone
does, with the integrals it leaves over still standing:

In [2]:
def sub_rules(value):
    """Every rule held in `value` - a rule, or a list/tuple of them."""
    if isinstance(value, Rule):
        yield value
    elif isinstance(value, (list, tuple)):
        for item in value:
            yield from sub_rules(item)


def children(rule):
    return [c for slot in rule._get_slots() for c in sub_rules(getattr(rule, slot))]


def stub(value):
    """The same value with every rule inside it replaced by a DontKnowRule,
    which evaluates to Integral(integrand, variable): a hole for a later
    step to fill."""
    if isinstance(value, Rule):
        return DontKnowRule(value.integrand, value.variable)
    if isinstance(value, list):
        return [stub(v) for v in value]
    if isinstance(value, tuple):
        return tuple(stub(v) for v in value)
    return value


def one_step(rule):
    """What this rule alone does, and whether it left anything behind.  A few
    rules (CyclicPartsRule) reach into their sub-rules and cannot be run half
    way: those do their work in a single step."""
    try:
        return type(rule)(*[stub(getattr(rule, s)) for s in rule._get_slots()]).eval(), True
    except Exception:
        return rule.eval(), False


one_step(integral_steps(x * sin(x), x))

(x*Integral(sin(x), x) - Integral(-cos(x), x), True)

## Flattening the tree

Now walk it.  Each rule fills one hole - the `Integral` it explains - with
what it produces, and its sub-rules fill the holes it leaves in turn, so the
recursion resolves the innermost integrals last and every step is one rule.

Two details that only show up once you try it:

- a rule whose hole is not in the expression has nothing to do here.  That is
  how `AlternativeRule` works out: SymPy takes its first alternative, which
  fills the hole, and the others then find nothing left to fill;
- a substitution ends by putting the original variable back, so `URule` gets
  a closing step of its own.

In [3]:
def find_hole(expr, rule):
    """The unevaluated integral in `expr` that `rule` explains, if it is
    still there."""
    for node in expr.atoms(Integral):
        if node.function == rule.integrand and list(node.variables) == [rule.variable]:
            return node
    return None


def rule_name(rule):
    name = type(rule).__name__
    return name[:-4] if name.endswith("Rule") else name


def flatten(rule, expr, out):
    hole = find_hole(expr, rule)
    if hole is None:
        return expr                       # nothing of this rule is left to do
    result, partial = one_step(rule)
    filled = expr.subs(hole, result)
    if filled != expr:
        out.append((filled, rule_name(rule)))
        expr = filled
    if partial:
        for child in children(rule):
            expr = flatten(child, expr, out)
    if isinstance(rule, URule):           # ... and put x back
        back = expr.subs(rule.u_var, rule.u_func)
        if back != expr:
            out.append((back, f"substitute back u = {rule.u_func}"))
            expr = back
    return expr


def integration_history(integrand, variable, title=None):
    """The whole derivation as a History: the integral, then one expression
    per rule, down to the antiderivative."""
    start = Integral(integrand, variable)
    steps = []
    flatten(integral_steps(integrand, variable), start, steps)
    return History([start] + steps, title=title or f"∫ {integrand} d{variable}")


history = integration_history(x * sin(x), x)
list(zip(history.actions, history.steps))

[(None, Integral(x*sin(x), x)),
 ('Parts', x*Integral(sin(x), x) - Integral(-cos(x), x)),
 ('Sin', -x*cos(x) - Integral(-cos(x), x)),
 ('ConstantTimes', -x*cos(x) + Integral(cos(x), x)),
 ('Cos', -x*cos(x) + sin(x))]

## Watching it

`display_history` puts the viewer in the cell: every step with what changed
in green, what the step before lost in red, and a **Play** button that runs
the derivation as a slideshow.

In [4]:
display_history(integration_history(x * sin(x), x))

A substitution, an alternative and a rewrite, all in one:

In [5]:
display_history(integration_history(sin(x) ** 3, x))

Partial fractions - six steps, no substitution:

In [ ]:
display_history(integration_history(1 / (x**2 - 1), x))

## Saving it

The same history can be written to a file that works offline - KaTeX and its
fonts travel with it - and plays there too:

In [ ]:
from sympy_editor import save_history_html

save_history_html(integration_history(atan(x), x), "integration_steps.html")

## Where the line is

Everything above is notebook code.  `sympy_editor` sees only a `History`: a
list of expressions and a word about what produced each.  The steps of an
integration, a chain of rewrites, the output of an algorithm of your own -
all the same to it.